# Customizing WhyLabs LLM Metrics

This notebook shows how you can customize the LLM Metrics that appear in the Whylabs LLM dashboard.

In [22]:
!pip install whylabs_toolkit


[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


Edit the following for your organization and model.

In [23]:
org_id = 'org-5Hsdjx'
dataset_id = 'model-98'
base_url = 'https://api.whylabsapp.com'

In [24]:
import getpass
api_key = getpass.getpass("Enter API Key:")

Set up the client API

In [25]:
import whylabs_client
from whylabs_client.api import models_api
from whylabs_client.model.metric_schema import MetricSchema
from whylabs_client.model.column_schema import ColumnSchema
configuration = whylabs_client.Configuration(
    host = base_url
)
configuration.api_key['ApiKeyAuth'] = api_key


api = models_api.ModelsApi(whylabs_client.ApiClient(configuration))

This first example overrides the default built-in metric used to visualize the 'prompt.sentiment_nltk' data, choosing instead to use the 75th percentile. The LLM metric name is set to 'majority_sentiment'. This is done by creating a custom metric using the WhyLabs models API.

In [26]:
col_to_change = 'prompt.sentiment_nltk'
name = "majority_sentiment"
results = api.put_entity_schema_metric(org_id, dataset_id, MetricSchema(
        name = name,
        label = "Majority sentiment",
        column=col_to_change,
        default_metric="quantile_75"))
schema = api.get_entity_schema(org_id, dataset_id)
schema.metrics[name]

{'builtin_metric': 'quantile_75',
 'column': 'prompt.sentiment_nltk',
 'default_metric': 'quantile_75',
 'label': 'Majority sentiment'}

The second example changes the tab that 'response.relevance_to_prompt' is displayed in. This is done by setting the 'tags' in the column schema for that data to include 'performance', 'security' or both.

In [27]:
col_to_move = 'response.relevance_to_prompt'
col_schema = api.get_entity_schema_column(org_id, dataset_id, col_to_move)
col_schema.tags = ['security']

results = api.put_entity_schema_column(org_id, dataset_id, col_to_move, col_schema)
api.get_entity_schema_column(org_id, dataset_id, col_to_move)

{'classifier': 'input',
 'data_type': 'fractional',
 'discreteness': 'continuous',
 'tags': ['security']}

The third example removes the 'prompt.has_patterns' from either of the dashboard tabs by setting tags to contain a value other than 'performance' or 'security'. If the tags array is empty, the default categorization of the metric will be restored.

In [28]:
col_to_remove = 'prompt.has_patterns'
col_schema = api.get_entity_schema_column(org_id, dataset_id, col_to_remove)
col_schema.tags = ['quality']

results = api.put_entity_schema_column(org_id, dataset_id, col_to_remove, col_schema)
api.get_entity_schema_column(org_id, dataset_id, col_to_remove)

{'classifier': 'input',
 'data_type': 'null',
 'discreteness': 'discrete',
 'tags': ['quality']}